# 04 · Structured Prompting
### Practical LLM Engineering — Module 02: Prompt Engineering

> **What you'll learn**
> - Why unstructured LLM output breaks production pipelines
> - JSON schema enforcement, XML tags, and markdown templates
> - Pydantic-based output validation and auto-retry loops
> - Function calling / tool use as structured output
> - Constrained decoding with grammars (outlines / llama.cpp)
> - Engineering patterns: parse-validate-retry, graceful degradation

---


## 1. Overview

In production LLM systems, **free-form text output is almost never acceptable**. Downstream code needs to parse, validate, and act on model responses reliably.

The failure modes without structured output:

```python
# What you asked for:
"Return JSON: {name: str, age: int, score: float}"

# What you sometimes get:
"Sure! Here is the JSON you requested:
```json
{
  'name': 'Alice',   ← single quotes — invalid JSON
  'age': 'thirty',  ← wrong type
  'score': null,    ← missing required field
}
```
Hope this helps!"  ← trailing text breaks json.loads()
```

**Structured prompting** is the set of techniques that make LLM output reliably machine-readable:

```
Structured Prompting Toolkit
├── 1. Schema-in-prompt   — describe format in the system prompt
├── 2. XML / delimiter    — wrap fields in named tags
├── 3. JSON mode          — API-level enforcement (Claude, OpenAI)
├── 4. Function calling   — declare a typed tool schema
├── 5. Pydantic + retry   — validate output, retry on failure
└── 6. Grammar-constrained decoding — token-level enforcement (local)
```

### Module position

```
01_zero_shot_prompting   ✅
02_few_shot_prompting    ✅
03_chain_of_thought      ✅
04_structured_prompting  ← you are here
05_prompt_evaluation
```


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install anthropic openai pydantic jsonschema -q

import os, re, json, time, textwrap
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, Any, Type, get_type_hints
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Pydantic v2
try:
    from pydantic import BaseModel, Field, ValidationError, field_validator
    import pydantic
    print(f"✅ Pydantic {pydantic.VERSION}")
except ImportError:
    print("⚠️  Run: pip install pydantic")

print("✅ Libraries ready")


In [ ]:
# ── LLM client abstraction (from notebook 01) ────────────────────────
@dataclass
class LLMResponse:
    text: str; model: str; input_tokens: int
    output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class BaseLLMClient(ABC):
    @abstractmethod
    def complete(self, system, user, max_tokens=512, temperature=0.0) -> LLMResponse: ...
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

class ClaudeClient(BaseLLMClient):
    def __init__(self, model="claude-sonnet-4-20250514", api_key=None):
        import anthropic
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.messages.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{"role":"user","content":user}])
        return LLMResponse(msg.content[0].text, self.model,
            msg.usage.input_tokens, msg.usage.output_tokens,
            (time.perf_counter()-t0)*1000)

class OpenAIClient(BaseLLMClient):
    def __init__(self, model="gpt-4o-mini", api_key=None):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.chat.completions.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role":"system","content":system},{"role":"user","content":user}])
        c = msg.choices[0]
        return LLMResponse(c.message.content, self.model,
            msg.usage.prompt_tokens, msg.usage.completion_tokens,
            (time.perf_counter()-t0)*1000)

class HuggingFaceClient(BaseLLMClient):
    def __init__(self, model="microsoft/Phi-3-mini-4k-instruct", device="auto"):
        from transformers import pipeline
        self.model_name = model
        self.pipe = pipeline("text-generation", model=model,
                              device_map=device, torch_dtype="auto")
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        msgs = [{"role":"system","content":system},{"role":"user","content":user}]
        t0   = time.perf_counter()
        out  = self.pipe(msgs, max_new_tokens=max_tokens,
                         temperature=max(temperature,1e-4),
                         do_sample=temperature>0, return_full_text=False)
        text = out[0]["generated_text"]
        if isinstance(text, list): text = text[-1]["content"]
        return LLMResponse(text, self.model_name, 0,
                           len(text.split()), (time.perf_counter()-t0)*1000)

def get_llm(backend="claude", **kwargs):
    return {"claude":ClaudeClient,"anthropic":ClaudeClient,
            "openai":OpenAIClient,"gpt":OpenAIClient,
            "hf":HuggingFaceClient,"local":HuggingFaceClient}[backend.lower()](**kwargs)

# ── Mock client ───────────────────────────────────────────────────────
import itertools

class MockLLMClient(BaseLLMClient):
    """Mock returning realistic structured outputs."""
    _VALID_JSON = json.dumps({
        "name": "Alice Chen", "age": 32, "role": "Senior Engineer",
        "skills": ["Python", "PyTorch", "distributed systems"],
        "score": 9.2, "available": True
    })
    _INVALID_JSON = "Sure! Here is the data: {'name': 'Alice', age: 32, 'score': null}"
    _XML = "<person><name>Alice Chen</name><age>32</age><role>Engineer</role></person>"
    _FUNC = json.dumps({
        "tool_name": "search_products",
        "parameters": {"query": "wireless headphones", "max_price": 150, "in_stock": True}
    })
    _responses = itertools.cycle([_VALID_JSON, _VALID_JSON, _INVALID_JSON,
                                   _VALID_JSON, _VALID_JSON])
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        time.sleep(0.03)
        lower = user.lower() + system.lower()
        if "xml" in lower or "<" in system:
            text = self._XML
        elif "function" in lower or "tool" in lower:
            text = self._FUNC
        elif "json" in lower or "{" in system:
            text = next(self._responses)
        else:
            text = "The structured output has been generated successfully."
        return LLMResponse(text, "mock-llm",
                           len((system+user).split()), len(text.split()), 35.0)

llm = MockLLMClient()
print("MockLLMClient active — swap for get_llm('claude') when ready.")


## 3. Background: The Structured Output Problem

### 3.1 Why LLMs produce malformed output

LLMs are trained to maximise the likelihood of *human-written* text. Humans rarely write raw JSON — they write natural language with occasional structured snippets, surrounded by prose.

This means:
- A model asked for JSON may add explanatory preamble
- It may use Python dict syntax (`'key'`) instead of JSON (`"key"`)
- It may hallucinate extra fields or omit required ones
- Field types may be wrong (string `"32"` instead of integer `32`)

### 3.2 The parse–validate–retry loop

The standard production pattern for reliable structured output:

```
┌─────────────────────────────────────────────────────┐
│  1. GENERATE    → LLM produces text                  │
│  2. EXTRACT     → find JSON/XML/etc. in response     │
│  3. PARSE       → json.loads() / xml.parse()         │
│  4. VALIDATE    → check schema (Pydantic, jsonschema) │
│  5. RETRY?      → if invalid: add error to prompt    │
│                   and call LLM again (max N retries)  │
│  6. FALLBACK    → if still invalid: return None      │
│                   or default, log the failure         │
└─────────────────────────────────────────────────────┘
```

### 3.3 Spectrum of enforcement strength

| Method | Enforcement | Requires |
|---|---|---|
| Schema in prompt | Soft (model may ignore) | Nothing extra |
| XML tags | Medium (easy to parse) | Nothing extra |
| JSON mode | Hard (API enforces valid JSON) | API support |
| Function calling | Hard (schema-validated) | API support |
| Grammar-constrained decoding | Absolute (token-level) | Local model |


## 4. Schema-in-Prompt

In [ ]:
# ── Technique 1: Embed the schema directly in the system prompt ────────

def schema_in_prompt(llm: BaseLLMClient, user_text: str,
                      schema: dict, strict: bool = False) -> dict:
    """
    Ask the LLM to produce output matching a given schema.
    schema: dict mapping field names to (type_str, description) tuples.
    """
    schema_lines = []
    for field_name, (type_str, desc) in schema.items():
        schema_lines.append(f'  "{field_name}": {type_str}  // {desc}')
    schema_str = "{\n" + ",\n".join(schema_lines) + "\n}"

    system = (
        "You are a precise data extraction assistant.\n"
        "Respond with a single JSON object matching this schema exactly:\n"
        f"{schema_str}\n\n"
        "Rules:\n"
        "- Output ONLY the JSON object, no markdown fences, no explanation\n"
        "- Use double quotes for all strings\n"
        "- Null for missing values\n"
        + ("- Every field is required\n" if strict else "")
    )
    resp = llm(system=system, user=user_text, max_tokens=300)
    return resp.text


# ── Demo: extract structured data from unstructured text ─────────────
CANDIDATE_TEXT = """
Hi, I'm Alice Chen. I have 8 years of experience in software engineering,
specialising in Python, PyTorch, and distributed systems. I'm currently
a Senior Engineer at TechCorp and am looking for ML infrastructure roles.
My hourly rate expectation is around $180. I'm available immediately.
"""

schema = {
    "name":           ("string",         "Full name"),
    "years_exp":      ("integer",        "Years of professional experience"),
    "current_role":   ("string",         "Current job title"),
    "skills":         ("array of string","Top technical skills"),
    "hourly_rate":    ("number",         "Expected hourly rate in USD"),
    "available_now":  ("boolean",        "Whether immediately available"),
}

raw = schema_in_prompt(llm, CANDIDATE_TEXT, schema, strict=True)
print("Schema-in-prompt output:")
print(raw)
print()
try:
    parsed = json.loads(raw)
    print("✅ Valid JSON")
    for k, v in parsed.items():
        print(f"   {k:<16}: {v!r}")
except json.JSONDecodeError as e:
    print(f"❌ JSON error: {e}")


## 5. Robust JSON Extraction

In [ ]:
# ── Handle the many ways LLMs produce JSON ────────────────────────────
def extract_json(text: str) -> Optional[Any]:
    """
    Robustly extract JSON from LLM output.
    Handles: fenced code blocks, prose wrapping, single quotes, trailing commas.
    Returns parsed object or None.
    """
    # 1. Strip markdown fences
    text = re.sub(r"```(?:json)?\s*", "", text).strip()
    text = re.sub(r"```\s*$", "", text).strip()

    # 2. Try direct parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 3. Extract first {...} or [...] block
    for pattern in [r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", r"\{.*\}", r"\[.*\]"]:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            candidate = match.group()
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass

    # 4. Fix common issues and retry
    fixed = text
    fixed = re.sub(r"'([^']+)'(\s*:)", r'"\1"\2', fixed)   # single-quoted keys
    fixed = re.sub(r":\s*'([^']*)'", r': "\1"',   fixed)   # single-quoted values
    fixed = re.sub(r",\s*([}\]])",    r"\1",       fixed)   # trailing commas
    fixed = re.sub(r"//[^\n]*",       "",          fixed)   # JS-style comments
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    return None


# ── Test on deliberately broken responses ─────────────────────────────
broken_responses = [
    # Type 1: Markdown fenced
    '```json\n{"name": "Alice", "age": 32}\n```',
    # Type 2: Prose wrapper
    'Here is the JSON:\n{"name": "Bob", "score": 9.5}\nHope that helps!',
    # Type 3: Single quotes
    "{'name': 'Carol', 'available': True}",
    # Type 4: Trailing comma
    '{"name": "Dave", "skills": ["Python", "ML",]}',
    # Type 5: JS comments
    '{"name": "Eve" // the candidate\n, "age": 28}',
    # Type 6: Completely unparseable
    "The answer is: name=Frank, age=45",
]

print("Robust JSON extraction:")
print(f"{'Input (truncated)':<50} {'Result'}")
print("-" * 75)
for resp in broken_responses:
    result = extract_json(resp)
    label  = "✅ parsed" if result else "❌ failed"
    preview = repr(resp[:45])
    print(f"{preview:<50} {label}  {str(result)[:40] if result else ''}")


## 6. Pydantic Validation + Auto-Retry

In [ ]:
# ── Define schemas with Pydantic ─────────────────────────────────────
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import List, Optional as Opt

class CandidateProfile(BaseModel):
    name:          str              = Field(..., description="Full name")
    years_exp:     int              = Field(..., ge=0, le=50)
    current_role:  str              = Field(..., description="Job title")
    skills:        List[str]        = Field(..., min_length=1, max_length=20)
    hourly_rate:   Opt[float]       = Field(None, ge=0, le=10_000)
    available_now: bool             = Field(...)
    seniority:     Opt[str]         = Field(None)

    @field_validator("seniority", mode="before")
    @classmethod
    def infer_seniority(cls, v, info):
        if v: return v
        yoe = info.data.get("years_exp", 0)
        if   yoe < 2:  return "junior"
        elif yoe < 6:  return "mid"
        elif yoe < 12: return "senior"
        else:          return "staff"

    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v):
        if not v.strip():
            raise ValueError("Name cannot be empty")
        return v.strip().title()


class ProductSearchParams(BaseModel):
    query:     str
    max_price: Opt[float] = None
    min_rating: Opt[float] = Field(None, ge=0.0, le=5.0)
    in_stock:  bool        = True
    category:  Opt[str]   = None


# ── Demonstrate validation ─────────────────────────────────────────────
good_data = {
    "name": "alice chen", "years_exp": 8, "current_role": "Senior Engineer",
    "skills": ["Python", "PyTorch"], "hourly_rate": 180.0, "available_now": True
}

bad_data = {
    "name": "", "years_exp": -1, "current_role": "Engineer",
    "skills": [], "available_now": "yes"   # wrong type
}

print("Pydantic validation:")
print()
for label, data in [("Good data", good_data), ("Bad data", bad_data)]:
    try:
        profile = CandidateProfile(**data)
        print(f"✅ {label}: {profile.model_dump()}")
    except ValidationError as e:
        errors = [f"{err['loc'][0]}: {err['msg']}" for err in e.errors()]
        print(f"❌ {label}: {errors}")
    print()


In [ ]:
# ── Parse-Validate-Retry loop ─────────────────────────────────────────
def structured_llm_call(
    llm:         BaseLLMClient,
    system:      str,
    user:        str,
    schema:      Type[BaseModel],
    max_retries: int = 3,
    max_tokens:  int = 512,
) -> tuple[Optional[BaseModel], dict]:
    """
    Call the LLM and parse its response into a Pydantic model.
    On validation failure, feed the error back to the LLM and retry.

    Returns: (parsed_model_or_None, metadata_dict)
    """
    schema_json = json.dumps(schema.model_json_schema(), indent=2)
    base_system = (
        f"{system}\n\n"
        f"Respond ONLY with a JSON object matching this schema:\n"
        f"{schema_json}\n"
        f"No markdown fences. No explanation. Pure JSON only."
    )

    history   = []
    meta      = {"attempts": 0, "errors": [], "total_tokens": 0}

    for attempt in range(max_retries):
        meta["attempts"] += 1

        # Build user message (add error context on retry)
        if history:
            last_err  = history[-1]["error"]
            retry_msg = (
                f"{user}\n\n"
                f"Your previous response caused this validation error:\n"
                f"{last_err}\n\n"
                f"Please fix the issue and return valid JSON only."
            )
        else:
            retry_msg = user

        resp = llm(system=base_system, user=retry_msg, max_tokens=max_tokens)
        meta["total_tokens"] += resp.total_tokens

        # Extract and parse
        parsed_raw = extract_json(resp.text)
        if parsed_raw is None:
            err = "Could not extract any JSON from response"
            meta["errors"].append(err)
            history.append({"response": resp.text, "error": err})
            continue

        # Validate with Pydantic
        try:
            model_instance = schema(**parsed_raw)
            meta["success"] = True
            return model_instance, meta
        except ValidationError as e:
            err = "; ".join(
                f"{'.'.join(str(l) for l in er['loc'])}: {er['msg']}"
                for er in e.errors()
            )
            meta["errors"].append(err)
            history.append({"response": resp.text, "error": err})

    meta["success"] = False
    return None, meta


# ── Run it ────────────────────────────────────────────────────────────
EXTRACT_SYSTEM = "You are a precise data extraction assistant."

result, meta = structured_llm_call(
    llm, EXTRACT_SYSTEM, CANDIDATE_TEXT,
    schema=CandidateProfile, max_retries=3
)

print("Parse-Validate-Retry result:")
print(f"  Attempts : {meta['attempts']}")
print(f"  Success  : {meta['success']}")
print(f"  Errors   : {meta['errors'] or 'none'}")
print(f"  Tokens   : {meta['total_tokens']}")
if result:
    print(f"  Output   : {result.model_dump()}")


## 7. XML Tags as Structure

XML tags are often **more reliable** than JSON for LLM output because:
- They allow nested free-form text without escaping issues
- The model can reason in natural language within tags
- Partial extraction is possible even if the response is incomplete
- Tags are more visible in the prompt (easier for the model to follow)


In [ ]:
# ── XML-based structured extraction ───────────────────────────────────
XML_SYSTEM = """You are a precise information extraction assistant.
Always respond using XML tags exactly as specified.
Include all requested tags even if the value is unknown (use <unknown/>)."""

def xml_extract(llm: BaseLLMClient, text: str, fields: list[str]) -> dict:
    """
    Extract fields from text using XML-tagged responses.
    fields: list of tag names to extract.
    """
    tags_spec = "\n".join(f"<{f}>value</{f}>" for f in fields)
    system = (
        f"{XML_SYSTEM}\n\n"
        f"Extract the following fields and return them inside these XML tags:\n"
        f"{tags_spec}"
    )
    resp = llm(system=system, user=f"Extract from:\n{text}", max_tokens=400)
    return parse_xml_tags(resp.text, fields)


def parse_xml_tags(text: str, tags: list[str]) -> dict:
    """Extract values from XML-tagged LLM response."""
    result = {}
    for tag in tags:
        pattern = rf"<{tag}>(.*?)</{tag}>"
        match   = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        result[tag] = match.group(1).strip() if match else None
    return result


# ── Demo on a job posting ─────────────────────────────────────────────
JOB_POSTING = """
We are looking for a Senior ML Engineer to join our Sydney-based team.
The role involves designing and deploying large-scale recommendation systems
using PyTorch and Kubernetes. Candidates should have 5+ years experience
with distributed training and MLOps. Salary range: $180k–$220k.
Must be eligible to work in Australia. Full-time, hybrid work model.
Applications close 30 June 2025.
"""

fields = ["job_title", "location", "required_skills", "years_experience",
           "salary_range", "work_model", "application_deadline"]

extracted = xml_extract(llm, JOB_POSTING, fields)

print("XML extraction from job posting:")
print()
for field, value in extracted.items():
    status = "✅" if value and value != "unknown" else "⚠️"
    print(f"  {status} {field:<25}: {value or 'not found'}")


In [ ]:
# ── XML vs JSON reliability comparison ───────────────────────────────
# Simulate parse success rates across 20 calls

formats = {
    "JSON (no fences)": {
        "responses": [
            '{"title": "Engineer", "salary": 180000}',
            '{"title": "Engineer", "salary": 180000}',
            "Here is the JSON: {'title': 'Engineer'}",  # bad
            '{"title": "Engineer", "salary": 180000}',
            '```json\n{"title": "Engineer"}\n```',     # fenced
        ] * 4,
    },
    "XML tags": {
        "responses": [
            "<title>Engineer</title><salary>180000</salary>",
            "<title>Engineer</title><salary>180000</salary>",
            "<title>Engineer</title>\n(salary not provided)",  # partial — still parseable
            "<TITLE>Engineer</TITLE><SALARY>180000</SALARY>",   # case variation
            "<title>Senior Engineer</title><salary>$180k</salary>",
        ] * 4,
    },
}

print("Parse success rate simulation (20 responses each):\n")
for fmt_name, data in formats.items():
    successes = 0
    for resp in data["responses"]:
        if "JSON" in fmt_name:
            successes += 1 if extract_json(resp) is not None else 0
        else:
            parsed = parse_xml_tags(resp, ["title", "salary"])
            successes += 1 if any(v for v in parsed.values()) else 0
    rate = successes / len(data["responses"])
    bar  = "█" * int(rate * 20) + "░" * (20 - int(rate * 20))
    print(f"  {fmt_name:<20} {bar}  {rate:.0%}")


## 8. Function Calling / Tool Use

Modern LLM APIs support **function calling** (OpenAI) or **tool use** (Anthropic) — a structured way to ask the model to fill in typed parameters for a predefined function schema.

This is the strongest form of schema enforcement at the API level:

```python
tools = [{
    "name": "search_products",
    "description": "Search the product catalogue",
    "input_schema": {
        "type": "object",
        "properties": {
            "query":     {"type": "string"},
            "max_price": {"type": "number"},
            "in_stock":  {"type": "boolean"}
        },
        "required": ["query"]
    }
}]
```

The API guarantees the response matches the declared schema — no parsing needed.


In [ ]:
# ── Simulate function calling output ─────────────────────────────────
# (Real implementation shown for Claude and OpenAI below)

TOOLS = [
    {
        "name":        "search_products",
        "description": "Search the product catalogue with filters",
        "parameters": {
            "type": "object",
            "properties": {
                "query":      {"type": "string",  "description": "Search query"},
                "max_price":  {"type": "number",  "description": "Maximum price in USD"},
                "min_rating": {"type": "number",  "description": "Minimum star rating (0-5)"},
                "in_stock":   {"type": "boolean", "description": "Only return in-stock items"},
                "category":   {"type": "string",  "description": "Product category"},
            },
            "required": ["query"],
        },
    },
    {
        "name":        "get_order_status",
        "description": "Check the status of a customer order",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id":    {"type": "string"},
                "customer_id": {"type": "string"},
            },
            "required": ["order_id"],
        },
    },
]


def simulate_tool_call(llm: BaseLLMClient, tools: list[dict],
                        user_query: str) -> dict:
    """
    Ask the LLM to choose a tool and fill in its parameters.
    Production: use native API tool_use; this simulates the output format.
    """
    tools_desc = "\n".join(
        f"- {t['name']}: {t['description']}\n"
        f"  Parameters: {json.dumps(t['parameters']['properties'], indent=2)}"
        for t in tools
    )
    system = (
        "You are a tool dispatcher. Given a user query, select the most appropriate "
        "tool and extract the required parameters.\n\n"
        f"Available tools:\n{tools_desc}\n\n"
        "Respond with a JSON object only:\n"
        '{"tool_name": "<name>", "parameters": {<key>: <value>, ...}}''
    )
    resp   = llm(system=system, user=user_query, max_tokens=200)
    parsed = extract_json(resp.text)
    if not parsed:
        return {"tool_name": None, "parameters": {}, "error": "parse_failed"}

    # Validate parameters against schema
    tool = next((t for t in tools if t["name"] == parsed.get("tool_name")), None)
    if not tool:
        return {"tool_name": None, "parameters": {}, "error": "unknown_tool"}

    required = tool["parameters"].get("required", [])
    missing  = [r for r in required if r not in parsed.get("parameters", {})]
    if missing:
        return {**parsed, "error": f"missing_required: {missing}"}

    return {**parsed, "error": None}


# ── Test tool dispatch ────────────────────────────────────────────────
test_queries = [
    "Find me wireless headphones under $150 that are in stock",
    "What's the status of order #ORD-2024-9871?",
    "Show me 4-star or higher laptops in the computing category",
    "I need kitchen appliances under $80",
]

print("Function calling / tool dispatch:")
print()
for query in test_queries:
    result = simulate_tool_call(llm, TOOLS, query)
    status = "✅" if result["error"] is None else f"❌ {result['error']}"
    print(f"  Query : {query[:55]}")
    print(f"  Tool  : {result.get('tool_name')}  {status}")
    print(f"  Params: {result.get('parameters', {})}")
    print()


In [ ]:
# ── Real Claude tool_use call (uncomment with API key) ───────────────
"""
import anthropic

client = anthropic.Anthropic()

# Define tool with Anthropic's schema format
claude_tools = [{
    "name": "search_products",
    "description": "Search the product catalogue",
    "input_schema": {
        "type": "object",
        "properties": {
            "query":     {"type": "string"},
            "max_price": {"type": "number"},
            "in_stock":  {"type": "boolean"},
        },
        "required": ["query"]
    }
}]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=claude_tools,
    messages=[{
        "role": "user",
        "content": "Find wireless headphones under $150 that are in stock"
    }]
)

# Response will contain a tool_use block if a tool was called
for block in response.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Input: {block.input}")  # Already parsed — no json.loads needed!
"""
print("Real tool_use: uncomment the code above and add your ANTHROPIC_API_KEY")
print()

# ── Real OpenAI function_call (uncomment with API key) ───────────────
"""
from openai import OpenAI
client = OpenAI()

openai_tools = [{
    "type": "function",
    "function": {
        "name": "search_products",
        "description": "Search the product catalogue",
        "parameters": {
            "type": "object",
            "properties": {
                "query":     {"type": "string"},
                "max_price": {"type": "number"},
                "in_stock":  {"type": "boolean"},
            },
            "required": ["query"]
        }
    }
}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Find wireless headphones under $150 in stock"}],
    tools=openai_tools,
    tool_choice="auto",
)

tool_call = response.choices[0].message.tool_calls[0]
print(f"Tool: {tool_call.function.name}")
print(f"Args: {json.loads(tool_call.function.arguments)}")
"""
print("OpenAI function calling: uncomment the block above with your OPENAI_API_KEY")


## 9. Grammar-Constrained Decoding

The strongest form of structured output: **constrain the token sampling** itself so the model *cannot* produce invalid output.

Libraries like **outlines** and **llama.cpp** implement this by:

1. At each generation step, computing which vocabulary tokens are valid given the partial output and the grammar
2. Masking invalid tokens to $-\infty$ before softmax
3. This guarantees 100% format compliance — not just probabilistically, but absolutely

```python
import outlines
import outlines.models as models
import outlines.generate as generate

model = models.transformers('mistralai/Mistral-7B-Instruct-v0.1')

# Define schema as Pydantic model
class ProductReview(BaseModel):
    product: str
    rating: int        # guaranteed to be int
    pros: List[str]    # guaranteed to be list
    cons: List[str]

# Generator is constrained to the schema
generator = generate.json(model, ProductReview)
review = generator('Write a review for AirPods Pro')
# review is *always* a valid ProductReview — no retries needed
```

**Tradeoff:** requires local model access (cannot use API models this way) and adds ~10-15% overhead to token generation.


In [ ]:
# ── Simulate constrained decoding behaviour ──────────────────────────
# (Real implementation needs 'outlines' library + local model)
# Here we demonstrate the conceptual token masking

import torch
import torch.nn.functional as F

def constrained_next_token(logits: torch.Tensor,
                            valid_token_ids: list[int]) -> int:
    """Mask all tokens except valid_token_ids, then sample.
    This is the core operation in grammar-constrained decoding.
    """
    mask = torch.full_like(logits, float("-inf"))
    mask[valid_token_ids] = 0.0
    constrained_logits = logits + mask
    probs = F.softmax(constrained_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Simulate: model is generating a JSON integer field
# e.g. "rating": _ where only digits 1-5 are valid
vocab_size = 50_000
fake_logits = torch.randn(vocab_size)

# Token IDs for digits 0-9 (approximate; real vocab varies)
digit_token_ids = list(range(16, 26))   # placeholder IDs for "0"-"9"

unconstrained = fake_logits.argmax().item()
constrained   = constrained_next_token(fake_logits, digit_token_ids)

print("Constrained decoding illustration:")
print(f"  Unconstrained argmax token ID : {unconstrained}")
print(f"  Constrained to digits [0-9]   : {constrained}  (always in valid set)")
print()

# ── Grammar types supported by outlines ──────────────────────────────
grammar_types = {
    "JSON schema":      "Generate JSON matching a Pydantic/jsonschema schema",
    "Regex":            "Generate text matching a regular expression pattern",
    "CFG (EBNF)":       "Generate text matching a context-free grammar",
    "Choice":           "Select from a fixed set of strings",
    "Integer / Float":  "Generate valid numbers only",
}

print("Grammar-constrained decoding — supported constraint types:")
for gtype, desc in grammar_types.items():
    print(f"  {gtype:<22} : {desc}")

print()
print("Libraries: outlines (local HF models), llama.cpp (GGUF models)")
print("           vLLM supports guided decoding in API mode")


## 10. Production Patterns

In [ ]:
# ── Pattern 1: Type coercion post-processing ────────────────────────
def coerce_types(raw: dict, schema: Type[BaseModel]) -> dict:
    """
    Attempt type coercion before Pydantic validation.
    Handles common LLM mistakes: string numbers, string booleans.
    """
    hints    = get_type_hints(schema)
    coerced  = {}
    for field_name, value in raw.items():
        if field_name not in hints:
            coerced[field_name] = value
            continue
        expected = hints[field_name]
        # Unwrap Optional[X] → X
        origin = getattr(expected, "__origin__", None)
        if origin is type(None):
            coerced[field_name] = value
            continue
        args = getattr(expected, "__args__", None)
        if args:
            expected = next((a for a in args if a is not type(None)), expected)

        try:
            if expected == int:
                coerced[field_name] = int(str(value).replace(",", "").strip())
            elif expected == float:
                coerced[field_name] = float(str(value).replace(",", "").strip())
            elif expected == bool:
                if isinstance(value, str):
                    coerced[field_name] = value.lower() in {"true", "yes", "1", "y"}
                else:
                    coerced[field_name] = bool(value)
            else:
                coerced[field_name] = value
        except (ValueError, TypeError):
            coerced[field_name] = value

    return coerced


# Test coercion
raw_with_type_errors = {
    "name": "Alice Chen",
    "years_exp": "8",          # string instead of int
    "current_role": "Engineer",
    "skills": ["Python"],
    "hourly_rate": "180.00",   # string instead of float
    "available_now": "yes",    # string instead of bool
}

coerced = coerce_types(raw_with_type_errors, CandidateProfile)
try:
    profile = CandidateProfile(**coerced)
    print("✅ Coercion + validation succeeded:")
    print(f"   years_exp    : {profile.years_exp!r} (type: {type(profile.years_exp).__name__})")
    print(f"   hourly_rate  : {profile.hourly_rate!r} (type: {type(profile.hourly_rate).__name__})")
    print(f"   available_now: {profile.available_now!r} (type: {type(profile.available_now).__name__})")
except ValidationError as e:
    print(f"❌ Still failed: {e}")


In [ ]:
# ── Pattern 2: Graceful degradation ─────────────────────────────────
@dataclass
class StructuredResult:
    """Container for structured LLM results with degradation support."""
    data:       Optional[BaseModel]
    raw_text:   str
    attempts:   int
    succeeded:  bool
    errors:     list[str]
    fallback:   Optional[dict] = None

    def get(self, field: str, default=None):
        """Access field with fallback to raw extraction or default."""
        if self.data:
            return getattr(self.data, field, default)
        if self.fallback and field in self.fallback:
            return self.fallback[field]
        return default


def structured_call_with_fallback(
    llm:        BaseLLMClient,
    system:     str,
    user:       str,
    schema:     Type[BaseModel],
    max_retries: int = 3,
    fallback_fields: list[str] = None,
) -> StructuredResult:
    """
    Full structured call pipeline with graceful degradation:
    1. Try Pydantic-validated extraction
    2. On failure: attempt regex fallback for key fields
    3. Always return a usable result
    """
    # Attempt full structured extraction
    model_instance, meta = structured_llm_call(
        llm, system, user, schema, max_retries
    )

    if model_instance:
        return StructuredResult(
            data=model_instance, raw_text=user,
            attempts=meta["attempts"], succeeded=True,
            errors=meta["errors"]
        )

    # Fallback: regex extraction of key fields
    fallback_data = {}
    if fallback_fields:
        resp = llm(system=system, user=user, max_tokens=300)
        for field_name in fallback_fields:
            # Simple regex: look for "field_name: value" patterns
            pattern = rf"{re.escape(field_name)}[:\s]+([\w\s,\.\$\-]+?)(?:[,\n]|$)"
            match   = re.search(pattern, resp.text, re.IGNORECASE)
            if match:
                fallback_data[field_name] = match.group(1).strip()

    return StructuredResult(
        data=None, raw_text=user,
        attempts=meta["attempts"], succeeded=False,
        errors=meta["errors"], fallback=fallback_data
    )


# Demo
result = structured_call_with_fallback(
    llm, EXTRACT_SYSTEM, CANDIDATE_TEXT,
    schema=CandidateProfile, max_retries=2,
    fallback_fields=["name", "years_exp", "current_role"]
)

print("Structured call with graceful degradation:")
print(f"  Succeeded : {result.succeeded}")
print(f"  Attempts  : {result.attempts}")
print(f"  Name      : {result.get('name', 'unknown')}")
print(f"  Role      : {result.get('current_role', 'unknown')}")
print(f"  Fallback  : {result.fallback}")


## 11. Engineering Guide: Choosing a Structured Output Method

In [ ]:
# ── Benchmark: parse success rate by method ──────────────────────────
import random

def simulate_parse_success(method: str, n: int = 50) -> float:
    """Simulate parse success rates based on empirical LLM behaviour."""
    rates = {
        "No structure (free text)":    0.30,
        "Schema in system prompt":     0.72,
        "JSON + extraction heuristics": 0.88,
        "XML tags":                    0.91,
        "Pydantic + retry (max=1)":    0.88,
        "Pydantic + retry (max=3)":    0.97,
        "Function calling (API)":      0.99,
        "Grammar-constrained (local)": 1.00,
    }
    p = rates.get(method, 0.5)
    return sum(random.random() < p for _ in range(n)) / n

random.seed(42)
methods = [
    "No structure (free text)",
    "Schema in system prompt",
    "JSON + extraction heuristics",
    "XML tags",
    "Pydantic + retry (max=1)",
    "Pydantic + retry (max=3)",
    "Function calling (API)",
    "Grammar-constrained (local)",
]

results_parse = {m: simulate_parse_success(m) for m in methods}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if v < 0.8 else "#f39c12" if v < 0.95 else "#2ecc71"
          for v in results_parse.values()]
bars = ax.barh(list(results_parse.keys()), list(results_parse.values()),
               color=colors, edgecolor="white", linewidth=0.8)
ax.set_xlabel("Parse success rate")
ax.set_title("Structured Output: Parse Success Rate by Method")
ax.set_xlim(0, 1.05)
ax.axvline(0.95, color="gray", linestyle="--", alpha=0.5, label="95% target")
ax.legend()
for bar, v in zip(bars, results_parse.values()):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
            f"{v:.0%}", va="center", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Decision guide ───────────────────────────────────────────────────
guide = {
    "Simple flat JSON, <5 fields":       ("Schema in prompt",     "Fast, cheap, works for simple cases"),
    "Complex nested JSON":               ("Pydantic + retry",     "Handles nesting, catches type errors"),
    "Mixed text + structured fields":    ("XML tags",             "Tolerates prose between fields"),
    "OpenAI / Claude API, typed params": ("Function calling",     "Strongest API-level guarantee"),
    "Local model, 100% required":        ("Grammar-constrained",  "outlines or llama.cpp grammar"),
    "High volume, latency sensitive":    ("Pydantic + coercion",  "Coerce first, validate, skip retries"),
    "Unreliable model (small LLM)":      ("Pydantic + retry(3)",  "Multiple retries with error feedback"),
}

print("Structured Prompting Decision Guide:")
print(f"{'Situation':<42} {'Method':<26} {'Notes'}")
print("-" * 95)
for situation, (method, notes) in guide.items():
    print(f"{situation:<42} {method:<26} {notes}")


## 12. Engineering Insights

### 12.1 Retry cost analysis

Each retry adds tokens and latency. Budget retries carefully:

$$
\text{expected cost} = \sum_{k=1}^{K} P(\text{fail})^{k-1} \cdot P(\text{success}) \cdot k \cdot C_{\text{call}}
$$

If the first-attempt success rate is $p$ and cost per call is $C$:
- $p = 0.9$: expected 1.11 calls
- $p = 0.7$: expected 1.43 calls
- $p = 0.5$: expected 2.00 calls

### 12.2 Version your schemas

Schemas evolve as requirements change. **Breaking schema changes** are a common source of production failures:

```python
# v1 schema
class Profile_v1(BaseModel):
    name: str
    score: float

# v2 — added required field (BREAKING)
class Profile_v2(BaseModel):
    name: str
    score: float
    tier: str   ← LLM prompt must be updated simultaneously

# Best practice: make new fields Optional initially
class Profile_v2_safe(BaseModel):
    name:  str
    score: float
    tier:  Optional[str] = None   ← safe migration
```

### 12.3 Structured output in RAG pipelines

Structured prompting is especially powerful in **Retrieval-Augmented Generation** pipelines where the LLM must return:
- Which retrieved passages it used (citations)
- Confidence scores
- Whether it could answer from context or not

Always use typed schemas for RAG answer objects to prevent hallucinated citations and malformed responses flowing downstream.


## 13. Exercises

1. **Retry curve:** Measure first-attempt parse success rate for your LLM backend on 20 structured extraction tasks. Then add retries (max=1, 2, 3) and plot cumulative success rate vs number of retries. At what retry count does adding more retries stop helping?

2. **Schema evolution:** Start with a `ProductReview` schema (product, rating, summary). Add two new fields across three versions. Implement a migration function that accepts any version of the parsed JSON and coerces it to the latest schema.

3. **XML vs JSON head-to-head:** Pick 10 complex extraction tasks (nested objects, lists, dates). Run both XML and JSON methods 3 times each. Compare parse success rate, field completeness, and token usage.

4. **Function calling pipeline:** Implement a simple agent that has 3 tools (search, get_details, add_to_cart). Given a user query like "Find me a blue running shoe under $100 and add it to my cart", chain multiple tool calls to complete the task.

5. **Constrained decoding simulation:** Implement a toy constrained decoder for a simple grammar: `rating ::= [1-5]` and `label ::= 'positive' | 'negative' | 'neutral'`. Show that masking invalid tokens changes the output distribution to always produce valid values.

---

## 14. Key Takeaways

| Concept | Summary |
|---|---|
| **The core problem** | LLMs produce prose; pipelines need typed data |
| **Extract-parse-validate-retry** | Standard production loop for robust structured output |
| **`extract_json()`** | Handles fences, single quotes, trailing commas |
| **Pydantic** | Schema definition + type coercion + validation in one |
| **XML tags** | More reliable than JSON for free-text fields; partial parse possible |
| **Function calling** | Strongest API-level enforcement; no post-processing needed |
| **Grammar-constrained** | Absolute enforcement at token level; requires local model |
| **Type coercion** | Coerce before validating to catch string/int/bool mismatches |
| **Graceful degradation** | Always return *something* usable; log, don't crash |
| **Schema versioning** | New required fields break old prompts — use Optional on migration |

---

**Next notebook →** `05_prompt_evaluation.ipynb` — building systematic evaluation pipelines to measure and compare prompt performance at scale.
